### Display clustering results
Notebook to compare two metrics for clustering: MTEB, which uses sklearn.cluster.MiniBatchKMeans() and kNN, which uses sklearn.neighbors.KNeighborsClassifier()
MTEB results are saved in .json files, one for each task. 
kNN results are saved in .json files, one for each model.
This notebook aggregates the results for multiple tasks and multiple models.

In [ ]:
import os
import mteb
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import regex as re
import math

In [ ]:
# get task type 
mteb.get_task("ArguAna").metadata.type

In [ ]:
mteb.get_task("ArguAna").metadata.name

#### Analyze MTEB results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/sparse_results/Tfidf"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    #print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df_mteb = pd.DataFrame.from_dict(data, orient='index')
df_mteb.index.name = "task_name"

In [ ]:
task_selection = ["ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P",
                 "RedditClusteringP2P", "StackExchangeClusteringP2P"]
model_exclusion = ["1.0", "svd", "svd_log_piecewise"]

df_mteb_c = df_mteb.loc[task_selection].drop(model_exclusion, axis=1)
df_mteb_c

#### Analyze kNN results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/knn_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
        
    for file in files:
        # Skip unwanted files
        model_name = file.strip(".json")
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                for key, value in json_data.items():
                    if type(value) == list:
                        value = np.mean(value)
                    json_data[key] = round(value*100, 2)
                
                
                data[model_name.removeprefix("Tfidf_").replace("old_", "")] = json_data
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df_knn = pd.DataFrame.from_dict(data)
df_knn

In [ ]:
task_order = ["ArxivClusteringP2P", "arxiv_full", "arxiv_batchwise", "BiorxivClusteringP2P", "biorxiv_full", "biorxiv_batchwise", 
              "MedrxivClusteringP2P", "medrxiv_full", "medrxiv_batchwise", "RedditClusteringP2P", "reddit_full", "reddit_batchwise",
              "StackExchangeClusteringP2P", "stackexchange_full", "stackexchange_batchwise"]
model_order = ["tfidf_log", "svd_log_novocab", "svd_log_old", "svd_log", "svd50_log", "svd200_log", "svd300_log", "svd500_log", "rnd100_log", "rnd768_log"]
df = pd.concat([df_mteb_c, df_knn])
df.index = df.index.str.strip()
df = df.reindex(task_order, columns=model_order)
df

In [ ]:
df.to_latex()

#### Plotting the results

In [ ]:
# add column specifying the task type
df["task_types"] = ["mteb", "kNN full", "kNN batched"]*5

In [ ]:
df.index.name = "task_name"

In [ ]:
df_melted = df.reset_index().melt(id_vars=["task_name", "task_types"], var_name="version", value_name="score")

In [ ]:
df_melted

In [ ]:
# write function to get dimensionality of model
def extract_number(s: str):
    """
    Extracts a number from a string and returns it as an integer.
    If no number is found, returns NaN.
    """
    
    match = re.search(r'\d+', s)
    if match:
        return int(match.group())
    else:
        return math.nan


In [ ]:
# write funciton to get dataset name
def dataset_name(s: str):
    s = s.lower()
    if "arxiv" in s:
        s = "arxiv"
    elif "bio" in s:
        s = "biorxiv"
    elif "med" in s:
        s = "medrxiv"
    elif "reddit" in s:
        s = "reddit"
    elif "stack" in s:
        s = "stackexchange"
    return s

In [ ]:
# write a function to get reduction type of model
def reduction_type(s: str):
    if "rnd" in s:
        return "random projection"
    elif "svd" in s:
        return "svd"
    else:
        return "no reduction" 

In [ ]:
df_melted["embedding dimensions"] = [extract_number(v) for v in df_melted["version"]]
df_melted["dataset"] = [dataset_name(n) for n in df_melted["task_name"]]
df_melted["reduction type"] = [reduction_type(v) for v in df_melted["version"]]

In [ ]:
df_melted

#### plot version 1
- 5 panels, 1 for each dataset
- x axis: embedding dims
- y axis: scores
- different colors for task types
- different opacities for reduction type? random proj at 50%

In [ ]:
# create color matching for task types
task_types = list(df_melted["task_types"].unique())
task_colors = dict(zip(task_types, sns.color_palette('colorblind', len(task_types))))

# create opacity matching for reduction type
opacity_dict = dict(zip(["random projection", "svd"], [0.5, 1.0]))